<a href="https://colab.research.google.com/github/thulasiswarreddy/DSA03-NLP/blob/main/NLP_assignment__01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
from collections import Counter

sentences = [
    "Book two tickets to Vellore",
    "What is the bus fare to Chennai",
    "Book one ticket to Madurai",
    "When does the train leave Chennai",
    "Tell me the bus timing to Vellore"
]

tokens = []

for sentence in sentences:
    words = re.findall(r'\b[a-zA-Z]+\b', sentence.lower())
    tokens.extend(words)

frequency = Counter(tokens)

print("Total Tokens:", len(tokens))
print("Unique Types:", len(frequency))

print("\nWord Frequencies:")
for word, count in frequency.items():
    print(word, ":", count)

for threshold in [1, 2, 3, 5]:
    vocabulary = {
        word for word, count in frequency.items()
        if count >= threshold
    }

    unknown = sum(
        count for word, count in frequency.items()
        if count < threshold
    )

    print("\nThreshold:", threshold)
    print("Vocabulary Size:", len(vocabulary))
    print("Tokens mapped to UNK:", unknown)


Total Tokens: 30
Unique Types: 21

Word Frequencies:
book : 2
two : 1
tickets : 1
to : 4
vellore : 2
what : 1
is : 1
the : 3
bus : 2
fare : 1
chennai : 2
one : 1
ticket : 1
madurai : 1
when : 1
does : 1
train : 1
leave : 1
tell : 1
me : 1
timing : 1

Threshold: 1
Vocabulary Size: 21
Tokens mapped to UNK: 0

Threshold: 2
Vocabulary Size: 6
Tokens mapped to UNK: 15

Threshold: 3
Vocabulary Size: 2
Tokens mapped to UNK: 23

Threshold: 5
Vocabulary Size: 0
Tokens mapped to UNK: 30


In [2]:
from collections import Counter

sentences = [
    ["book", "two", "tickets", "to", "vellore"],
    ["book", "one", "ticket", "to", "chennai"],
    ["what", "is", "the", "bus", "fare"],
    ["train", "to", "madurai"],
]

unigrams = Counter()
bigrams = Counter()
trigrams = Counter()

for sentence in sentences:
    words = ["<s>"] + sentence + ["</s>"]

    unigrams.update(words)

    for i in range(len(words) - 1):
        bigrams[(words[i], words[i+1])] += 1

    for i in range(len(words) - 2):
        trigrams[
            (words[i], words[i+1], words[i+2])
        ] += 1

print("Unigram Counts:", unigrams)
print("\nBigram Counts:", bigrams)
print("\nTrigram Counts:", trigrams)

def bigram_probability(w1, w2):
    return bigrams[(w1, w2)] / unigrams[w1]

def trigram_probability(w1, w2, w3):
    context_count = bigrams[(w1, w2)]

    if context_count == 0:
        return 0

    return trigrams[(w1, w2, w3)] / context_count

print("\nBigram Probability:")
print(bigram_probability("to", "vellore"))

print("\nTrigram Probability:")
print(trigram_probability("book", "two", "tickets"))

Unigram Counts: Counter({'<s>': 4, '</s>': 4, 'to': 3, 'book': 2, 'two': 1, 'tickets': 1, 'vellore': 1, 'one': 1, 'ticket': 1, 'chennai': 1, 'what': 1, 'is': 1, 'the': 1, 'bus': 1, 'fare': 1, 'train': 1, 'madurai': 1})

Bigram Counts: Counter({('<s>', 'book'): 2, ('book', 'two'): 1, ('two', 'tickets'): 1, ('tickets', 'to'): 1, ('to', 'vellore'): 1, ('vellore', '</s>'): 1, ('book', 'one'): 1, ('one', 'ticket'): 1, ('ticket', 'to'): 1, ('to', 'chennai'): 1, ('chennai', '</s>'): 1, ('<s>', 'what'): 1, ('what', 'is'): 1, ('is', 'the'): 1, ('the', 'bus'): 1, ('bus', 'fare'): 1, ('fare', '</s>'): 1, ('<s>', 'train'): 1, ('train', 'to'): 1, ('to', 'madurai'): 1, ('madurai', '</s>'): 1})

Trigram Counts: Counter({('<s>', 'book', 'two'): 1, ('book', 'two', 'tickets'): 1, ('two', 'tickets', 'to'): 1, ('tickets', 'to', 'vellore'): 1, ('to', 'vellore', '</s>'): 1, ('<s>', 'book', 'one'): 1, ('book', 'one', 'ticket'): 1, ('one', 'ticket', 'to'): 1, ('ticket', 'to', 'chennai'): 1, ('to', 'chennai', 

In [3]:
from collections import Counter

sentences = [
    ["book", "ticket", "to", "vellore"],
    ["book", "ticket", "to", "chennai"],
    ["check", "bus", "fare"],
]

bigrams = Counter()

for sentence in sentences:
    for i in range(len(sentence) - 1):
        bigrams[(sentence[i], sentence[i+1])] += 1

vocabulary = set(
    word for sentence in sentences for word in sentence
)

V = len(vocabulary)

def laplace(w1, w2):
    count = bigrams[(w1, w2)]
    context = sum(
        count for (a, b), count in bigrams.items()
        if a == w1
    )

    return (count + 1) / (context + V)

def add_k(w1, w2, k=0.1):
    count = bigrams[(w1, w2)]
    context = sum(
        count for (a, b), count in bigrams.items()
        if a == w1
    )

    return (count + k) / (context + k * V)

print("Laplace Probability:",
      laplace("book", "ticket"))

print("Add-k Probability:",
      add_k("book", "ticket"))

print("Unseen Bigram Probability:",
      laplace("book", "fare"))

frequency_of_frequencies = Counter(bigrams.values())

print("Good-Turing Frequency Table:")
for r, nr in sorted(frequency_of_frequencies.items()):
    print("N(", r, ") =", nr)


Laplace Probability: 0.3
Add-k Probability: 0.7500000000000001
Unseen Bigram Probability: 0.1
Good-Turing Frequency Table:
N( 1 ) = 4
N( 2 ) = 2


In [4]:
def unigram_prob(word, uni, total):
    return uni[word] / total

def bigram_prob(w1, w2, bi, uni):
    if uni[w1] == 0:
        return 0
    return bi[(w1, w2)] / uni[w1]

def interpolation(p1, p2, p3):
    l1 = 0.2
    l2 = 0.3
    l3 = 0.5

    return l1*p1 + l2*p2 + l3*p3

def backoff(p3, p2, p1):
    if p3 > 0:
        return p3
    elif p2 > 0:
        return 0.4 * p2
    else:
        return 0.4 * 0.4 * p1

p1 = 0.05
p2 = 0.10
p3 = 0.20

print("Interpolation Probability:",
      interpolation(p1, p2, p3))

print("Backoff Score:",
      backoff(0, p2, p1))

Interpolation Probability: 0.14
Backoff Score: 0.04000000000000001


In [5]:
import math

def entropy(probabilities):
    result = 0

    for p in probabilities:
        if p > 0:
            result -= p * math.log2(p)

    return result

def cross_entropy(actual, predicted):
    total = 0

    for p in predicted:
        total -= math.log2(max(p, 1e-10))

    return total / len(actual)

def perplexity(cross_entropy_value):
    return 2 ** cross_entropy_value

distribution = [0.5, 0.3, 0.2]

H = entropy(distribution)

print("Entropy:", H)

actual = ["bus", "to", "vellore"]
predicted = [0.5, 0.4, 0.6]

CE = cross_entropy(actual, predicted)

print("Cross Entropy:", CE)
print("Perplexity:", perplexity(CE))

Entropy: 1.4854752972273344
Cross Entropy: 1.0196312296845227
Perplexity: 2.0274006651911334


In [6]:
from collections import Counter

sentences = [
    ["book", "two", "tickets", "to", "Vellore"],
    ["what", "is", "the", "bus", "fare"],
    ["train", "to", "Chennai", "tomorrow"]
]

tagset = {
    "book": "VERB",
    "two": "NUM",
    "tickets": "NOUN",
    "to": "PREP",
    "Vellore": "LOC",
    "what": "WH",
    "is": "AUX",
    "the": "DET",
    "bus": "TRANSPORT",
    "fare": "NOUN",
    "train": "TRANSPORT",
    "Chennai": "LOC",
    "tomorrow": "DATE"
}

tag_counts = Counter()

for sentence in sentences:
    for word in sentence:
        tag = tagset.get(word, "NOUN")
        tag_counts[tag] += 1
        print(word, "->", tag)

print("\nTag Frequencies:")
print(tag_counts)

book -> VERB
two -> NUM
tickets -> NOUN
to -> PREP
Vellore -> LOC
what -> WH
is -> AUX
the -> DET
bus -> TRANSPORT
fare -> NOUN
train -> TRANSPORT
to -> PREP
Chennai -> LOC
tomorrow -> DATE

Tag Frequencies:
Counter({'NOUN': 2, 'PREP': 2, 'LOC': 2, 'TRANSPORT': 2, 'VERB': 1, 'NUM': 1, 'WH': 1, 'AUX': 1, 'DET': 1, 'DATE': 1})


In [7]:
def rule_based_tagger(words):

    tags = []

    locations = ["Chennai", "Vellore", "Madurai"]
    prepositions = ["to", "from", "at", "in"]
    determiners = ["the", "a", "an"]
    pronouns = ["I", "you", "we"]
    verbs = ["book", "check", "cancel", "tell"]
    transport = ["bus", "train"]

    for word in words:

        if word in locations:
            tag = "PROPN"

        elif word in prepositions:
            tag = "PREP"

        elif word in determiners:
            tag = "DET"

        elif word in pronouns:
            tag = "PRON"

        elif word in verbs:
            tag = "VERB"

        elif word in transport:
            tag = "TRANSPORT"

        elif word.isdigit():
            tag = "NUM"

        else:
            tag = "NOUN"

        tags.append((word, tag))

    return tags


sentence = ["book", "two", "tickets", "to", "Vellore"]

result = rule_based_tagger(sentence)

for word, tag in result:
    print(word, ":", tag)

book : VERB
two : NOUN
tickets : NOUN
to : PREP
Vellore : PROPN


In [8]:
import math

training = [
    (["book", "tickets", "to", "Vellore"],
     ["VERB", "NOUN", "PREP", "PROPN"]),

    (["check", "the", "bus", "fare"],
     ["VERB", "DET", "NOUN", "NOUN"]),

    (["train", "to", "Chennai"],
     ["NOUN", "PREP", "PROPN"])
]

transition = {}
emission = {}
tag_counts = {}

for words, tags in training:

    previous = "<START>"

    for word, tag in zip(words, tags):

        transition.setdefault(previous, {})
        transition[previous][tag] = (
            transition[previous].get(tag, 0) + 1
        )

        emission.setdefault(tag, {})
        emission[tag][word] = (
            emission[tag].get(word, 0) + 1
        )

        tag_counts[tag] = tag_counts.get(tag, 0) + 1

        previous = tag

states = list(tag_counts.keys())

def viterbi(words):

    V = [{}]
    path = {}

    for tag in states:
        trans_count = transition["<START>"].get(tag, 0)
        trans_total = sum(transition["<START>"].values())

        trans_p = (trans_count + 0.1) / (
            trans_total + 0.1 * len(states)
        )

        emit_p = (
            emission[tag].get(words[0], 0) + 0.1
        ) / (
            tag_counts[tag] + 0.1 * 100
        )

        V[0][tag] = math.log(trans_p) + math.log(emit_p)
        path[tag] = [tag]

    for i in range(1, len(words)):

        V.append({})
        new_path = {}

        for tag in states:

            candidates = []

            for previous in states:

                trans_total = sum(
                    transition.get(previous, {}).values()
                )

                trans_p = (
                    transition.get(previous, {}).get(tag, 0) + 0.1
                ) / (trans_total + 0.1 * len(states))

                emit_p = (
                    emission[tag].get(words[i], 0) + 0.1
                ) / (tag_counts[tag] + 0.1 * 100)

                score = (
                    V[i-1][previous]
                    + math.log(trans_p)
                    + math.log(emit_p)
                )

                candidates.append((score, previous))

            best_score, best_previous = max(candidates)

            V[i][tag] = best_score
            new_path[tag] = path[best_previous] + [tag]

        path = new_path

    best_tag = max(V[-1], key=V[-1].get)

    return path[best_tag]


sentence = ["book", "tickets", "to", "Vellore"]

print("Predicted Tags:", viterbi(sentence))

Predicted Tags: ['VERB', 'NOUN', 'PREP', 'PROPN']


In [9]:
words = ["book", "tickets", "to", "Vellore"]

initial_tags = ["NOUN", "NOUN", "PREP", "NOUN"]

correct_tags = ["VERB", "NOUN", "PREP", "PROPN"]

def accuracy(predicted, actual):
    correct = sum(
        p == a for p, a in zip(predicted, actual)
    )

    return correct / len(actual) * 100

print("Initial Tags:", initial_tags)
print("Initial Accuracy:",
      accuracy(initial_tags, correct_tags))

# Transformation rules

rules = [
    ("NOUN", "VERB", "book"),
    ("NOUN", "PROPN", "Vellore")
]

for old_tag, new_tag, target_word in rules:

    for i, word in enumerate(words):

        if word == target_word and initial_tags[i] == old_tag:
            initial_tags[i] = new_tag

print("\nCorrected Tags:", initial_tags)

print("Final Accuracy:",
      accuracy(initial_tags, correct_tags))

Initial Tags: ['NOUN', 'NOUN', 'PREP', 'NOUN']
Initial Accuracy: 50.0

Corrected Tags: ['VERB', 'NOUN', 'PREP', 'PROPN']
Final Accuracy: 100.0


In [10]:
import time
from collections import Counter

sentences = [
    ["book", "two", "tickets", "to", "Vellore"],
    ["check", "the", "bus", "fare"],
    ["train", "to", "Chennai"],
    ["book", "one", "ticket", "to", "Madurai"]
]

start = time.perf_counter()

word_counts = Counter()

for sentence in sentences:
    word_counts.update(sentence)

training_time = time.perf_counter() - start

print("Word Frequencies:")
print(word_counts)

print("\nTraining Time:",
      training_time, "seconds")

print("Vocabulary Size:",
      len(word_counts))

# Testing the model

test_sentence = ["book", "tickets", "to", "Vellore"]

for word in test_sentence:

    if word in word_counts:
        print(word, "-> Available")
    else:
        print(word, "-> Unknown")

# Calculate frequency-based accuracy
correct = 0

predicted = ["VERB", "NOUN", "PREP", "PROPN"]
actual = ["VERB", "NOUN", "PREP", "PROPN"]

for p, a in zip(predicted, actual):
    if p == a:
        correct += 1

accuracy = correct / len(actual) * 100

print("\nTagging Accuracy:", accuracy, "%")

Word Frequencies:
Counter({'to': 3, 'book': 2, 'two': 1, 'tickets': 1, 'Vellore': 1, 'check': 1, 'the': 1, 'bus': 1, 'fare': 1, 'train': 1, 'Chennai': 1, 'one': 1, 'ticket': 1, 'Madurai': 1})

Training Time: 0.00010531700002047728 seconds
Vocabulary Size: 14
book -> Available
tickets -> Available
to -> Available
Vellore -> Available

Tagging Accuracy: 100.0 %


In [11]:
models = {
    "Unigram": {
        "perplexity": 120,
        "speed": 1000
    },

    "Bigram": {
        "perplexity": 85,
        "speed": 800
    },

    "Trigram": {
        "perplexity": 65,
        "speed": 600
    }
}

print("Language Model Comparison")

for name, values in models.items():

    print("\nModel:", name)
    print("Perplexity:", values["perplexity"])
    print("Speed:", values["speed"], "tokens/sec")

taggers = {
    "Rule-Based": 80,
    "HMM": 88,
    "Brill": 90
}

print("\nPOS Tagger Comparison")

for name, score in taggers.items():

    print(name, "Accuracy:", score, "%")

best_tagger = max(taggers, key=taggers.get)

print("\nHighest Example Accuracy:", best_tagger)

Language Model Comparison

Model: Unigram
Perplexity: 120
Speed: 1000 tokens/sec

Model: Bigram
Perplexity: 85
Speed: 800 tokens/sec

Model: Trigram
Perplexity: 65
Speed: 600 tokens/sec

POS Tagger Comparison
Rule-Based Accuracy: 80 %
HMM Accuracy: 88 %
Brill Accuracy: 90 %

Highest Example Accuracy: Brill
